## Generate GP curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
from typing import Sequence

### Utility functions

In [ ]:
type ArrayLike = NDArray | Sequence[int | float]

def exp_kernel(A: NDArray, B: NDArray, ls: float) -> NDArray:
    # RBF kernel: k(xa, xb) = exp(-0.5 * ||xa - xb||^2) (lengthscale=1)
    A = np.asarray(A)  # (n_a, n_features)
    B = np.asarray(B)  # (n_b, n_features)

    # squared euclidean distance matrix (n_a, n_b)
    sqdist = np.sum((A[:, None, :] - B[None, :, :]) ** 2, axis=-1)

    return np.exp(-0.5 * sqdist / (ls**2))


def power_spectral_density(omega: NDArray, ls: float | NDArray, n_dims: int) -> NDArray:
    """
    Power spectral density for the ExpQuad kernel.

    .. math::

        S(\\boldsymbol\\omega) =
            (\\sqrt(2 \\pi)^D \\prod_{i}^{D}\\ell_i
            \\exp\\left( -\\frac{1}{2} \\sum_{i}^{D}\\ell_i^2 \\omega_i^{2} \\right)

    Args:
        omega: array of shape (m_star, d), frequencies at which to evaluate the PSD.
        ls:    lengthscale(s), either a scalar or array of shape (d,).
        n_dims: number of input dimensions D.

    Returns:
        Array of shape (m_star,), one PSD value per basis function.
    """
    ls_arr = np.ones(n_dims) * ls          # (d,)
    c = np.power(np.sqrt(2.0 * np.pi), n_dims)
    exp = np.exp(-0.5 * np.dot(np.square(omega), np.square(ls_arr)))  # (m_star,)
    return c * np.prod(ls_arr) * exp       # (m_star,)

def calc_eigenvalues(L: ArrayLike, m: int, d: int) -> NDArray:
    """
    Calculate eigenvalues of the Laplacian on [-L1,L1] x ... x [-Ld,Ld]
    with Dirichlet boundary conditions, returning the m smallest.

    Parameters
    ----------
    L : NDArray, shape (d,)
        Domain half-widths per dimension.
    m : int
        Number of eigenfunctions to return.
    d : int
        Number of input dimensions.

    Returns
    -------
    selected_per_dim_eigenvalues : NDArray, shape (m,d)
        The m smallest eigenvalues, sorted ascending.
    """
    L = np.asarray(L, dtype=float)
    
    # Number of indices per dimension, scaled by relative domain size
    N_per_dim = np.ceil(m ** (1 / d) * L / L.min()).astype(int)

    # Build full multi-index grid (Cartesian product of per-dim indices)
    temp = [np.arange(1, 1 + N_per_dim[dim]) for dim in range(d)]
    grids = np.meshgrid(*temp, indexing='ij')
    NN = np.vstack([g.ravel() for g in grids]).T      # (N_total, d)

    # Compute all eigenvalues
    per_dim_eigvals = np.square((np.pi * NN) / (2 * L)) # (N_total, d)
    all_eigenvalues = np.sum(per_dim_eigvals, axis=1)

    # Sort and keep the m smallest
    sort_idx = np.argsort(all_eigenvalues)[:m]
    selected_per_dim_eigenvalues = per_dim_eigvals[sort_idx]
    # NN = NN[sort_idx]

    return selected_per_dim_eigenvalues

def calc_eigenvectors(Xs: NDArray, L: ArrayLike, per_dim_eigvals: NDArray) -> NDArray:
    """Calculate eigenvectors of the Laplacian.
    These are used as basis vectors in the HSGP approximation.

    Parameters
    ----------
    Xs              : NDArray, shape (n_samples, d)
    L               : NDArray, shape (d,)
    per_dim_eigvals : NDArray, shape (m, d)
        Per dimension eigenvalues, with the smallest sum

    Returns
    -------
    phi : NDArray, shape (n_samples, m)
    """
    L = np.asarray(L, dtype=float)
    
    # (1, m, d) * (n_samples, 1, d) -> (n_samples, m, d)
    term1 = np.sqrt(per_dim_eigvals)[None, :, :]          # (1, m, d)
    term2 = Xs[:, None, :] + L[None, None, :]     # (n_samples, 1, d) + (1, 1, d)
    c = 1.0 / np.sqrt(L)                          # (d,)

    phi = c * np.sin(term1 * term2)               # (n_samples, m, d)
    return np.prod(phi, axis=-1)                  # (n_samples, m)

### Initialize parameters

In [ ]:
ls = 0.3
var_f = 1.0
var_n = 0.01
N_train = 50
N_plot = 200
n_dim  = 1

L = [1]
m = 50
margin = 1.1

### Generate measurements from a 1D GP

$$
y_i = f(x_i) + \varepsilon_i \quad  \varepsilon_i \sim \mathcal{N}(0, \sigma_n^2)
$$

Where 
$$
f \sim \mathcal{GP}(0, k(x,x'))
$$

using the Squared Exponential kernel $k(x,x') = \sigma_f^2 e^ {-\frac{||x - x'||^2}{2 \ell^2}}$

In [ ]:
# Generate the test_test_inputs X*
x_train = np.linspace(-L[0],L[0],N_train).reshape(-1, 1) # (n_samples, d)
x_plot = np.linspace(-2,2,N_plot).reshape(-1, 1) # (n_samples, d)

# Sample latent function f jointly at both plot and training points
x_all = np.vstack([x_plot, x_train])
K_all = var_f * exp_kernel(x_all, x_all, ls)

# Sample the smooth latent function
rng = np.random.default_rng(seed=5)
f_all = rng.multivariate_normal(np.zeros(len(x_all)), K_all)

# Split into plot and train portions
f_plot = f_all[:N_plot]
f_at_train = f_all[N_plot:]

# Add noise to get observations y
y_train = f_at_train + rng.normal(0, np.sqrt(var_n), size=N_train)

# Compute standard deviation for the confidence band
K_ff = var_f*exp_kernel(x_plot, x_plot, ls)
std_f = np.sqrt(np.diag(K_ff))

fig, axs = plt.subplots(1, 1)
axs.fill_between(x_plot.flatten(), -2*std_f, 2*std_f, alpha=0.3, color='gray', label='±2σ of f')
axs.plot(x_plot.flatten(), f_plot)
axs.scatter(x_train.flatten(), y_train, c='r')
axs.legend()
plt.show()

### Compare performance of exact GP and HSGP

In [ ]:
# Compute posterior from exact GP
x_test = x_plot

K_yy = var_f * exp_kernel(x_train, x_train, ls) + np.eye(N_train) * var_n
K_yy_inv = np.linalg.solve(K_yy, np.eye(N_train))

K_sx = var_f * exp_kernel(x_test, x_train, ls)
K_xs = var_f * exp_kernel(x_train, x_test, ls)
K_ss = var_f * exp_kernel(x_test, x_test, ls)

gp_cov = K_ss - K_sx @ K_yy_inv @ K_xs
gp_mean = K_sx @ K_yy_inv @ y_train


$$
\hat{f_*} = \Phi _* (\sigma_n^2 \Lambda^{-1} + \Phi^\top \Phi)^{-1} y
$$

$$
cov(f_*) = \sigma_n^2\Phi _* (\sigma_n^2 \Lambda^{-1} + \Phi^\top \Phi)^{-1} \Phi^\top_*
$$

In [ ]:

# Compute posterior of HSGP
eigvals = calc_eigenvalues(L, m, n_dim)  # type: ignore # (m_start, d)
phi = calc_eigenvectors(x_train, L, eigvals)  # (n_samples, m_star)
phi_star = calc_eigenvectors(x_test, L, eigvals)
omega = np.sqrt(eigvals)  # (m_star, d)
psd = power_spectral_density(omega, ls, n_dim)

Lambda = np.diag(psd)                                        # (m_star, m_star)
A = var_n * np.linalg.inv(Lambda) + phi.T @ phi         # (m_star, m_star)
alpha = np.linalg.solve(A, phi.T @ y_train)                # (m_star,)
hsgp_mean = phi_star @ alpha                                # (n_test,)

# Predictive covariance
A_inv = np.linalg.inv(A)
hsgp_cov = var_n * phi_star @ A_inv @ phi_star.T        # (n_test, n_test)  


In [ ]:

gp_std = np.diag(gp_cov)
hsgp_std = np.diag(hsgp_cov)
print(hsgp_std)             

fig, axs = plt.subplots(1, 1)
# axs.plot(x_plot.flatten(), f_plot)
axs.plot(x_test.flatten(), gp_mean)
axs.plot(x_test.flatten(), hsgp_mean)
axs.scatter(x_train.flatten(), y_train, c='r', s=0.5)
axs.fill_between(x_test.flatten(),gp_mean -2*gp_std, gp_mean + 2*gp_std, alpha=0.3, color='gray', label='±2σ')
axs.fill_between(x_test.flatten(),hsgp_mean -20*hsgp_std, hsgp_mean + 20*hsgp_std, alpha=0.3, color='blue', label='±2σ')
plt.show()

#### Plot the GP prior

$$
\bm{f}_* \sim \mathrm{N}(\bm{0}, K(X_*, X_*))
$$

In [ ]:
rng = np.random.default_rng(seed=3)

# Generate the test_test_inputs X*
x_train = np.linspace(-5,5,200)


# compute covariance for the test_inputs
K = exp_kernel(x_train, x_train)
gp_mean = np.zeros(x_train.shape)

outputs = []
for i in range(3):
    outputs.append(
        rng.multivariate_normal(mean=gp_mean,cov=K)
    )

# Plot samle curves
for i in range(3):
    plt.plot(x_train, outputs[i])

# Plot covariance enveloppe 2 time the standard deviation
gp_std = np.sqrt(np.diag(K))
plt.fill_between(x_train, -2*gp_std, 2*gp_std, alpha=0.3, color='gray', label='±2σ')
plt.legend()
plt.xlabel('Input')
plt.ylabel('Output')
plt.title('GP Prior Samples with ±2σ Envelope')
plt.show()

#### Plot the GP posterior

$$
\begin{bmatrix}
\bm{f}\\ \bm{f}_*
\end{bmatrix} \sim
\mathrm{N}
\begin{pmatrix}
\bm{0}, & 
\begin{bmatrix}
K(X,X)& K(X,X_*)\\
K(X_*, X)& K(X_*,X_*)
\end{bmatrix}
\end{pmatrix}
$$

In [ ]:
training_inputs = np.linspace(start=-5, stop=5, num=5)
training_outputs = rng.normal(loc=0, scale=np.min(gp_std), size=training_inputs.shape)

K_XX = exp_kernel(training_inputs, training_inputs)
print(f"{K_XX.shape = }")
K_X_X_ = exp_kernel(x_train, x_train)
print(f"{K_X_X_.shape = }")
K_X_X = exp_kernel(x_train, training_inputs)
print(f"{K_X_X.shape = }")
K_XX_ = exp_kernel(training_inputs, x_train)
print(f"{K_XX_.shape = }")


K_inv = np.linalg.solve(K_XX, np.eye(N=K_XX.shape[0]))

gp_cov = K_X_X_ - K_X_X @ K_inv @ K_XX_
gp_mean = K_X_X @ K_inv @ training_outputs

# Generate plots
test_outputs = []
for i in range(3):
    test_outputs.append(
        rng.multivariate_normal(mean=gp_mean,cov=gp_cov)
    )

# Plot samle curves
for i in range(3):
    plt.plot(x_train, test_outputs[i])

plt.scatter(training_inputs, training_outputs)

# Plot std 
# Plot covariance enveloppe 2 time the standard deviation
gp_std = np.sqrt(np.diag(gp_cov))
plt.fill_between(x_train, -2*gp_std + gp_mean, 2*gp_std + gp_mean, alpha=0.3, color='gray', label='±2σ')
plt.legend()
plt.xlabel('Input')
plt.ylabel('Output')
plt.title('GP Prior Samples with ±2σ Envelope')
plt.show()

## 3 Tied down wiener filter

In [ ]:
def min_kernel(A:NDArray, B:NDArray):
    return np.minimum(A[:, None], B[None, :])

def min_prediction_kernel(A:NDArray, B:NDArray):
    return min_kernel(A,B) - A[:, None] @ B[None, :]


x_train = np.linspace(0,1,200)
training_inputs = np.array([1])
training_outputs = np.array([0])

gp_cov = min_prediction_kernel(x_train, x_train)
gp_mean = np.zeros(shape=x_train.shape)

# reset generator 
seed = 2
rng = np.random.default_rng(seed=seed)

# Sample predicitons using analytical solve
test_outputs = []
for i in range(3):
    test_outputs.append(
        rng.multivariate_normal(mean=gp_mean,cov=gp_cov)
    )

# Plot samle curves
for i in range(3):
    plt.plot(x_train, test_outputs[i])

plt.scatter(training_inputs, training_outputs)

# Plot covariance enveloppe 2 tim e the standard deviation
gp_std = np.sqrt(np.diag(gp_cov))
plt.fill_between(x_train, -2*gp_std + gp_mean, 2*gp_std + gp_mean, alpha=0.3, color='gray', label='±2σ')
plt.legend()
plt.xlabel('Input')
plt.ylabel('Output')
plt.title('GP Prior Samples with ±2σ Envelope')
plt.show()

In [ ]:
# Now using numerical solve
K_XX = min_kernel(training_inputs, training_inputs)
print(f"{K_XX.shape = }")
K_X_X_ = min_kernel(x_train, x_train)
print(f"{K_X_X_.shape = }")
K_X_X = min_kernel(x_train, training_inputs)
print(f"{K_X_X.shape = }")
K_XX_ = min_kernel(training_inputs, x_train)
print(f"{K_XX_.shape = }")


K_inv = np.linalg.solve(K_XX, np.eye(N=K_XX.shape[0]))

gp_cov = K_X_X_ - K_X_X @ K_inv @ K_XX_
gp_mean = K_X_X @ K_inv @ training_outputs

# reset generator 
seed = 2
rng = np.random.default_rng(seed=seed)

# Sample predicitons using analytical solve
test_outputs = []
for i in range(3):
    test_outputs.append(
        rng.multivariate_normal(mean=gp_mean,cov=gp_cov)
    )

# Plot samle curves
for i in range(3):
    plt.plot(x_train, test_outputs[i])

plt.scatter(training_inputs, training_outputs)


### 4.5.1

In [ ]:
import sympy
from sympy import exp, latex, invert, simplify

x1, x2, x3, x4, ell, x = sympy.symbols('x1 x2 x3 x4 ell x')

K = sympy.Matrix(
    [
        [1, exp(-(x2-x1)/ell), exp(-(x3-x1)/ell), exp(-(x4-x1)/ell)],
        [exp(-(x2-x1)/ell), 1, exp(-(x3-x2)/ell), exp(-(x4-x2)/ell)],
        [exp(-(x3-x1)/ell), exp(-(x3-x2)/ell), 1, exp(-(x4-x3)/ell)],
        [exp(-(x4-x1)/ell), exp(-(x4-x2)/ell), exp(-(x4-x3)/ell), 1],
    ]
)
# K_inv = simplify(K.inv())
# print(f"{latex(K_inv)}", )

# K_xX = sympy.Matrix(
#     [
#         exp(-(x-x1)/ell), exp(-(x-x2)/ell),exp(-(x3-x)/ell),exp(-(x4-x)/ell)
#     ]
# )
# w = simplify(K_xX.T * K_inv)
# print(f"{latex(w)}")




$\left[\begin{matrix}- \frac{e^{\frac{2 x_{2}}{\ell}}}{e^{\frac{2 x_{1}}{\ell}} - e^{\frac{2 x_{2}}{\ell}}} & \frac{e^{\frac{x_{1} + x_{2}}{\ell}}}{e^{\frac{2 x_{1}}{\ell}} - e^{\frac{2 x_{2}}{\ell}}} & 0 & 0\\\frac{e^{\frac{x_{1} + x_{2}}{\ell}}}{e^{\frac{2 x_{1}}{\ell}} - e^{\frac{2 x_{2}}{\ell}}} & \frac{\left(e^{\frac{2 x_{1}}{\ell}} - e^{\frac{2 x_{3}}{\ell}}\right) e^{\frac{2 x_{2}}{\ell}}}{e^{\frac{4 x_{2}}{\ell}} - e^{\frac{2 \left(x_{1} + x_{2}\right)}{\ell}} + e^{\frac{2 \left(x_{1} + x_{3}\right)}{\ell}} - e^{\frac{2 \left(x_{2} + x_{3}\right)}{\ell}}} & \frac{e^{\frac{x_{2} + x_{3}}{\ell}}}{e^{\frac{2 x_{2}}{\ell}} - e^{\frac{2 x_{3}}{\ell}}} & 0\\0 & \frac{e^{\frac{x_{2} + x_{3}}{\ell}}}{e^{\frac{2 x_{2}}{\ell}} - e^{\frac{2 x_{3}}{\ell}}} & \frac{\left(- e^{\frac{2 x_{2}}{\ell}} + e^{\frac{2 x_{4}}{\ell}}\right) e^{\frac{2 x_{3}}{\ell}}}{- e^{\frac{4 x_{3}}{\ell}} + e^{\frac{2 \left(x_{2} + x_{3}\right)}{\ell}} - e^{\frac{2 \left(x_{2} + x_{4}\right)}{\ell}} + e^{\frac{2 \left(x_{3} + x_{4}\right)}{\ell}}} & \frac{e^{\frac{x_{3} + x_{4}}{\ell}}}{e^{\frac{2 x_{3}}{\ell}} - e^{\frac{2 x_{4}}{\ell}}}\\0 & 0 & \frac{e^{\frac{x_{3} + x_{4}}{\ell}}}{e^{\frac{2 x_{3}}{\ell}} - e^{\frac{2 x_{4}}{\ell}}} & \frac{1}{1 - e^{\frac{2 \left(x_{3} - x_{4}\right)}{\ell}}}\end{matrix}\right]$

$\left[\begin{matrix}0 & \frac{\left(e^{\frac{2 x}{\ell}} - e^{\frac{2 x_{3}}{\ell}}\right) e^{\frac{- x + x_{2}}{\ell}}}{e^{\frac{2 x_{2}}{\ell}} - e^{\frac{2 x_{3}}{\ell}}} & \frac{\left(- e^{\frac{2 x}{\ell}} + e^{\frac{2 x_{2}}{\ell}}\right) e^{\frac{- x + x_{3}}{\ell}}}{e^{\frac{2 x_{2}}{\ell}} - e^{\frac{2 x_{3}}{\ell}}} & 0\end{matrix}\right]$

In [ ]:
a,b,c,d,e,f,g,h,j,k = sympy.symbols("a,b,c,d,e,f,g,h,j,k")
Kinv_test = sympy.Matrix(
    [
        [a, b, 0, 0],
        [c, d, e, 0],
        [0, f, g, h],
        [0, 0, j, k],
    ]
)
w = simplify(K_xX.T * Kinv_test)
print(f"{latex(w)}")


$\left[\begin{matrix}a e^{\frac{- x + x_{1}}{\ell}} + c e^{\frac{- x + x_{2}}{\ell}} & b e^{\frac{- x + x_{1}}{\ell}} + d e^{\frac{- x + x_{2}}{\ell}} + f e^{\frac{x - x_{3}}{\ell}} & e e^{\frac{- x + x_{2}}{\ell}} + g e^{\frac{x - x_{3}}{\ell}} + j e^{\frac{x - x_{4}}{\ell}} & h e^{\frac{x - x_{3}}{\ell}} + k e^{\frac{x - x_{4}}{\ell}}\end{matrix}\right]$

### 4.5.2 
Computer exercise: write code to draw samples from the neural network
covariance function, eq. (4.29) in 1-d and 2-d. Consider the cases when
var(u0 ) is either 0 or non-zero. Explain the form of the plots obtained
when var(u0 ) = 0.

In [ ]:
x_train = np.linspace(start=-5, stop=5, num=500).reshape(1,-1)

def nn_kernel(A:NDArray, B:NDArray, sigma:NDArray)->NDArray :
    """
    A : D x N
    B : D x N
    sigma : (D+1) x (D+1)
    """
    # Augment the input vectors
    ones = np.ones_like(a=A, shape=(1,A.shape[1]))
    A_augment = np.vstack([ones, A]) # (D+1) x N
    B_augment = np.vstack([ones, B]) # (D+1) x N

    numerator = 2 * A_augment.T @ sigma @ B_augment   # (N x N)
    A_self = 1 + 2 * np.diag(A_augment.T @ sigma @ A_augment).reshape(-1,1)
    B_self = 1 + 2 * np.diag(B_augment.T @ sigma @ B_augment).reshape(-1,1)

    denominator = np.sqrt(A_self @ B_self.T)

    return 2/np.pi * np.arcsin(numerator / denominator)

sigma = np.diag([1,10])
K_1d = nn_kernel(x_train, x_train, sigma)
gp_mean = np.zeros(x_train.flatten().shape)

# Visualize the kernel matrix
plt.figure(figsize=(8, 6))
plt.contour(K_1d,levels=([-0.5, 0 ,0.5 ,0.95]))
# plt.imshow(K_1d, cmap='viridis')
plt.colorbar(label='Kernel value')
plt.title('Neural Network Kernel Matrix (2D)')
plt.xlabel('Test point index')
plt.ylabel('Test point index')
plt.show()

In [ ]:
# reset generator 
seed = 2
rng = np.random.default_rng(seed=seed)

# Sample predicitons using analytical solve
sigma_zeros = [0,1]
test_outputs = []
for s0 in sigma_zeros :
    sigma = np.diag([s0, 10])

    gp_cov = nn_kernel(x_train, x_train, sigma)
    test_outputs.append(
        rng.multivariate_normal(mean=gp_mean,cov=gp_cov)
    )

# Plot samle curves
for test_output in test_outputs:
    plt.plot(x_train.flatten(), test_output)



In [ ]:
# For 2D
# Create a 2D grid of points: shape (2, N) where each column is a 2D coordinate
N = 20
x = np.linspace(0, 1, N)
y = np.linspace(0, 1, N)
xx, yy = np.meshgrid(x, y)

# Flatten and stack to create 2xN array (D=2, N=25)
test_inputs_2d = np.vstack([xx.flatten(), yy.flatten()])
print(f"test_inputs_2d shape: {test_inputs_2d.shape}")

# Define sigma for 2D case (3x3 for D=2, since we augment with bias)
sigma_2d = np.eye(3)

# Compute the kernel
K_2d = nn_kernel(test_inputs_2d, test_inputs_2d, sigma_2d)
print(f"K_2d shape: {K_2d.shape}")

# Visualize the kernel matrix
plt.figure(figsize=(8, 6))
plt.imshow(K_2d, cmap='viridis')
plt.colorbar(label='Kernel value')
plt.title('Neural Network Kernel Matrix (2D)')
plt.xlabel('Test point index')
plt.ylabel('Test point index')
plt.show()


In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Sample predicitons using analytical solve
sigma_zeros = [0,10]
test_outputs = []
for s0 in sigma_zeros :
    sigma = np.diag([s0, 10, 10])
    gp_mean = np.zeros(shape=(N*N,))

    gp_cov = nn_kernel(test_inputs_2d, test_inputs_2d, sigma)
    print(gp_cov.shape)
    test_outputs.append(
        rng.multivariate_normal(mean=gp_mean,cov=gp_cov)
    )


# Plot 3D surfaces for each sample
fig = plt.figure(figsize=(12, 5))

for idx, test_output in enumerate(test_outputs):
    ax = fig.add_subplot(1, 2, idx+1, projection='3d')
    
    # Reshape output back to 2D grid (N x N)
    output_grid = test_output.reshape(N, N)
    
    # Create mesh grid
    X = x
    Y = y
    
    # Plot surface
    surf = ax.plot_surface(xx, yy, output_grid, cmap='viridis', alpha=0.8)
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Output')
    ax.set_title(f'NN Samples (σ[0,0] = {sigma_zeros[idx]})')
    fig.colorbar(surf, ax=ax, shrink=0.5)

plt.tight_layout()
plt.show()
